# Notebook 02 — Build the Older-Adult Depression and Anxiety Cohort

This notebook converts the raw MIMIC-IV hospital and discharge-note tables
into one stable, labelled cohort for depression and anxiety modelling in
adults aged 60 years or older.

It performs the following steps:

1. Locate and validate the required MIMIC-IV files.
2. Calculate age at admission and retain older-adult admissions.
3. Derive depression and anxiety labels from ICD-9 and ICD-10 codes.
4. Stream discharge summaries in memory-safe chunks.
5. Lightly clean while preserving each complete discharge summary.
6. Keep one discharge summary per admission.
7. Create fixed patient-level train, validation and test splits.
8. Save the master cohort and aggregate audit reports.

The complete note is preserved so later notebooks can create both full-text
and selected-section representations. This notebook does **not** perform
model-specific section selection, tokenisation, truncation or chunking.


## Important data-access note

MIMIC-IV is credentialed clinical data. Do not commit the raw dataset,
individual clinical notes or generated patient-level cohort files to GitHub.
Each authorised team member should obtain the data through PhysioNet and keep
the same local project folder structure.


## 1. Import libraries


In [1]:
import gc
import json
import re
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

from IPython.display import display


## 2. Configure paths and cohort settings

The code searches upward from the current working directory for the project
repository, so it does not contain a team member's username or an absolute
machine-specific path.

Use `RUN_MODE = "smoke"` for a small end-to-end test. After every cell runs
successfully, change it to `"full"` and rerun the notebook from the beginning.

`SECTION_PROFILE` is intentionally fixed to `"none"`. This preserves the
complete note. The seven-section `clinical` profile and four-section `minimal`
profile are applied later in Notebook 03 when `text_selected` is created.

`TOKENIZER_NAME` is optional and should normally remain `None`. The resulting
`n_tokens` value is only a note-length estimate; actual model tokenisation is
performed in the model notebooks.


In [2]:
# ============================================================
# CELL 2 - CONFIGURE MACHINE-INDEPENDENT PATHS AND SETTINGS
# ============================================================

def find_project_root(starting_path):
    starting_path = Path(starting_path).expanduser().resolve()

    for location in [starting_path, *starting_path.parents]:
        if (location / ".git").exists() or (location / "dataset").exists():
            return location

    raise FileNotFoundError(
        "Could not locate the project root. Open Jupyter from inside "
        "the project repository."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
DATASET_DIR = PROJECT_ROOT / "dataset"
HOSP_DIR = DATASET_DIR / "hosp"
NOTE_DIR = DATASET_DIR / "note"


# Start with "smoke". Change to "full" only after the smoke run succeeds.
RUN_MODE = "full"


if RUN_MODE not in {"smoke", "full"}:
    raise ValueError("RUN_MODE must be either 'smoke' or 'full'.")


if RUN_MODE == "smoke":
    OUTPUT_DIR = DATASET_DIR / "cohort_output_smoke"
    SAMPLE_SIZE = 5_000
else:
    OUTPUT_DIR = DATASET_DIR / "cohort_output"
    SAMPLE_SIZE = None


# Cohort settings
MIN_AGE = 60
CHUNK_SIZE = 10_000
SEED = 42

# Keep this as "none". Section selection belongs in Notebook 03.
SECTION_PROFILE = "none"

TOKENIZER_NAME = None
EXCLUDE_BIPOLAR = False
KEEP_TEMPORARY_FILE = False


if SECTION_PROFILE != "none":
    raise ValueError(
        "Notebook 02 must preserve complete notes. "
        "Keep SECTION_PROFILE='none' and create selected text in Notebook 03."
    )


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TEMP_NOTES_PATH = OUTPUT_DIR / "_notes_tmp.parquet"


configuration = pd.DataFrame(
    {
        "setting": [
            "Project root",
            "Hospital tables",
            "Note tables",
            "Run mode",
            "Output directory",
            "Minimum age",
            "Chunk size",
            "Section profile",
            "Reference tokenizer",
            "Sample size",
            "Exclude bipolar admissions",
            "Random seed",
        ],
        "value": [
            PROJECT_ROOT,
            HOSP_DIR,
            NOTE_DIR,
            RUN_MODE,
            OUTPUT_DIR,
            MIN_AGE,
            CHUNK_SIZE,
            SECTION_PROFILE,
            TOKENIZER_NAME,
            SAMPLE_SIZE,
            EXCLUDE_BIPOLAR,
            SEED,
        ],
    }
)


display(configuration)


,setting,value
0,Project root,/Users/purnaperamune/Documents/GitHub/P000464S...
1,Hospital tables,/Users/purnaperamune/Documents/GitHub/P000464S...
2,Note tables,/Users/purnaperamune/Documents/GitHub/P000464S...
3,Run mode,full
4,Output directory,/Users/purnaperamune/Documents/GitHub/P000464S...
5,Minimum age,60
6,Chunk size,10000
7,Section profile,none
8,Reference tokenizer,None
9,Sample size,None


## 3. Define labels, leakage patterns and output columns

The labels are derived only from the specified ICD code families. Medication
mentions and explicit condition terms are measured for audit purposes; they do
not determine the labels.


In [3]:
# --------------------------------------------------------------------------
# 1. LABEL DEFINITIONS  -- core codes only, no extensions, no inference
# --------------------------------------------------------------------------

DEPRESSION = {
    "icd9":  (r"^2962", r"^2963", r"^3004", r"^311"),
    "icd10": (r"^F32", r"^F33", r"^F341"),
}

ANXIETY = {
    "icd9":  (r"^3000", r"^3002"),
    "icd10": (r"^F40", r"^F41"),
}

# Not a label. Only used if --exclude-bipolar is passed.
BIPOLAR = (r"^2960", r"^2961", r"^2964", r"^2965", r"^2966", r"^2967", r"^F31")

# Reported covariate only -- never contributes to a label.
# Restricted to agents prescribed almost exclusively for depression or anxiety.
# Deliberately excluded, with reason:
#   amitriptyline / nortriptyline / doxepin  -> neuropathic pain, migraine, sleep
#   mirtazapine                              -> appetite stimulation, sleep
#   bupropion                                -> smoking cessation
#   duloxetine                               -> neuropathic pain, fibromyalgia
#   trazodone                                -> insomnia (dominant inpatient use)
#   all benzodiazepines                      -> sedation, withdrawal, procedures
ANTIDEPRESSANT_PATTERN = re.compile(
    r"\b(?:sertraline|fluoxetine|paroxetine|citalopram|escitalopram|fluvoxamine|"
    r"venlafaxine|desvenlafaxine|vortioxetine|vilazodone|buspirone|"
    r"phenelzine|tranylcypromine|"
    r"zoloft|prozac|paxil|celexa|lexapro|luvox|effexor|pristiq|trintellix|"
    r"viibryd|buspar|nardil|parnate)\b",
    re.IGNORECASE,
)


DEP_LEAK_PATTERN = re.compile(
    r"\b(?:depress\w*|dysthymi\w*|MDD|major depressive)\b",
    re.IGNORECASE,
)

ANX_LEAK_PATTERN = re.compile(
    r"\b(?:anxi\w*|panic|phobi\w*|agoraphob\w*|GAD|generali[sz]ed anxiety)\b",
    re.IGNORECASE,
)

# Union, used for masking at training time. Both labels are masked together so
# a multi-label model cannot read either condition off the text.
LEAK_PATTERN = re.compile(
    f"(?:{DEP_LEAK_PATTERN.pattern})|(?:{ANX_LEAK_PATTERN.pattern})",
    re.IGNORECASE,
)

NOTE_COLS = ["note_id", "subject_id", "hadm_id", "note_type", "charttime", "text"]

DEID_PATTERN = re.compile(r"_{2,}|\[\*\*.*?\*\*\]")
WS_PATTERN = re.compile(r"[ \t]+")
NL_PATTERN = re.compile(r"\n{3,}")

FINAL_COLS = ["hadm_id", "subject_id", "age_at_admission", "gender",
              "depression", "anxiety", "n_tokens", "text"]


## 4. Define text-cleaning helpers

Cleaning removes null characters, standardises line endings and compresses
repeated whitespace without flattening the note. Section headings and complete
note content are preserved for Notebook 03 and the long-context models.


In [4]:
# ============================================================
# CELL 4 - DEFINE TEXT-CLEANING AND CODE-MATCHING HELPERS
# ============================================================


def compile_set(patterns):
    return re.compile("|".join(patterns))


def match_codes(df, icd9_pats, icd10_pats):
    '''Match ICD-9 patterns only to ICD-9 rows and ICD-10 patterns only
    to ICD-10 rows.'''
    out = np.zeros(len(df), dtype=bool)
    is9 = (df["icd_version"] == 9).to_numpy()

    if is9.any():
        out[is9] = (
            df.loc[is9, "code_clean"]
            .str.match(compile_set(icd9_pats), na=False)
            .to_numpy()
        )

    if (~is9).any():
        out[~is9] = (
            df.loc[~is9, "code_clean"]
            .str.match(compile_set(icd10_pats), na=False)
            .to_numpy()
        )

    return out


def clean_text(value):
    '''Apply light cleaning while preserving the complete note.'''
    if not isinstance(value, str):
        return ""

    value = value.replace("\x00", " ")
    value = value.replace("\r\n", "\n").replace("\r", "\n")
    value = DEID_PATTERN.sub("[DEID]", value)
    value = WS_PATTERN.sub(" ", value)
    value = NL_PATTERN.sub("\n\n", value)

    return value.strip()


def locate(directory, name):
    for extension in (".csv.gz", ".csv"):
        path = Path(directory) / f"{name}{extension}"
        if path.exists():
            return path

    raise FileNotFoundError(
        f"{name}.csv[.gz] was not found in {directory}"
    )


print("Text-cleaning and ICD-matching helpers were defined.")


Text-cleaning and ICD-matching helpers were defined.


## 5. Define admission and ICD-label loading functions


In [5]:
# --------------------------------------------------------------------------
# 2. SMALL TABLES
# --------------------------------------------------------------------------

def load_admissions(hosp_dir, min_age):
    print("[1/6] patients + admissions", flush=True)
    patients = pd.read_csv(
        locate(hosp_dir, "patients"),
        usecols=["subject_id", "gender", "anchor_age", "anchor_year"],
    )
    admissions = pd.read_csv(
        locate(hosp_dir, "admissions"),
        usecols=["subject_id", "hadm_id", "admittime"],
        parse_dates=["admittime"],
    )
    adm = admissions.merge(patients, on="subject_id", how="inner")
    # Per-admission age, not patient-level anchor_age.
    adm["age_at_admission"] = (
        adm["anchor_age"] + (adm["admittime"].dt.year - adm["anchor_year"])
    )
    n0 = len(adm)
    adm = adm[adm["age_at_admission"] >= min_age].copy()
    # anchor_age is masked at 91 for anyone over 89 -- flag for the report only
    adm["age_capped"] = adm["age_at_admission"] >= 91
    print(f"      {n0:,} admissions -> {len(adm):,} at age >= {min_age}", flush=True)
    return adm[["subject_id", "hadm_id", "gender", "age_at_admission", "age_capped"]]


def load_labels(hosp_dir, keep_hadm, exclude_bipolar):
    print("[2/6] diagnoses -> binary labels", flush=True)
    dx = pd.read_csv(
        locate(hosp_dir, "diagnoses_icd"),
        usecols=["hadm_id", "icd_code", "icd_version"],
        dtype={"icd_code": str},
    )
    dx = dx[dx["hadm_id"].isin(keep_hadm)].copy()
    dx["code_clean"] = (
        dx["icd_code"].str.upper().str.replace(".", "", regex=False).str.strip()
    )

    dx["depression"] = match_codes(dx, DEPRESSION["icd9"], DEPRESSION["icd10"])
    dx["anxiety"] = match_codes(dx, ANXIETY["icd9"], ANXIETY["icd10"])

    cols = ["depression", "anxiety"]
    matched = {
        lab: dx.loc[dx[lab], "code_clean"].value_counts().head(40).to_dict()
        for lab in cols
    }

    if exclude_bipolar:
        dx["bipolar"] = dx["code_clean"].str.match(compile_set(BIPOLAR), na=False)
        cols = cols + ["bipolar"]

    labels = dx.groupby("hadm_id")[cols].any()
    del dx
    gc.collect()
    return labels.reset_index(), matched


## 6. Define memory-safe discharge-note streaming

The discharge table is read in chunks. Complete cleaned notes are written
immediately to a temporary Parquet file while smaller numeric metadata remains
in memory. No section-selection logic is applied in this notebook.


In [6]:
# ============================================================
# CELL 6 - DEFINE MEMORY-SAFE DISCHARGE-NOTE STREAMING
# ============================================================


def stream_notes(
    note_dir,
    keep_hadm,
    tmp_path,
    chunksize=10_000,
    tokenizer_name=None,
    sample=None,
):
    '''Read discharge summaries in chunks and preserve complete notes.'''
    print("[3/6] discharge summaries (streaming)", flush=True)
    print("      text representation: complete cleaned note", flush=True)

    path = locate(note_dir, "discharge")
    keep = set(keep_hadm)

    tok = None
    if tokenizer_name:
        from transformers import AutoTokenizer

        tok = AutoTokenizer.from_pretrained(tokenizer_name)
        print(f"      reference tokenizer: {tokenizer_name}", flush=True)

    writer = None
    meta_parts = []
    seen = 0
    kept = 0

    reader = pd.read_csv(
        path,
        usecols=NOTE_COLS,
        chunksize=chunksize,
        dtype={"text": str},
    )

    try:
        for batch_number, chunk in enumerate(reader, start=1):
            seen += len(chunk)
            sub = chunk.loc[chunk["hadm_id"].isin(keep)].copy()
            del chunk

            if sample is not None:
                remaining = sample - kept
                if remaining <= 0:
                    break
                sub = sub.head(remaining).copy()

            if not sub.empty:
                sub["text"] = sub["text"].map(clean_text)
                sub["n_words"] = sub["text"].str.split().str.len()
                sub["char_len"] = sub["text"].str.len()

                sub["on_antidepressant"] = sub["text"].str.contains(
                    ANTIDEPRESSANT_PATTERN,
                    na=False,
                )
                sub["n_leak_depression"] = sub["text"].str.count(
                    DEP_LEAK_PATTERN
                )
                sub["n_leak_anxiety"] = sub["text"].str.count(
                    ANX_LEAK_PATTERN
                )

                if tok is not None:
                    encoded = tok(
                        sub["text"].tolist(),
                        truncation=False,
                        add_special_tokens=True,
                    )
                    sub["n_tokens"] = [
                        len(token_ids)
                        for token_ids in encoded["input_ids"]
                    ]
                    del encoded
                else:
                    # Reference estimate only; exact tokenisation happens later.
                    sub["n_tokens"] = (
                        sub["n_words"] * 1.4
                    ).round().astype(int)

                table = pa.Table.from_pandas(
                    sub[["hadm_id", "note_id", "text"]],
                    preserve_index=False,
                )

                if writer is None:
                    writer = pq.ParquetWriter(
                        tmp_path,
                        table.schema,
                        compression="snappy",
                    )

                writer.write_table(table)
                meta_parts.append(sub.drop(columns=["text"]))
                kept += len(sub)

                del sub, table
                gc.collect()

            if batch_number % 5 == 0:
                print(
                    f"      scanned {seen:,}, kept {kept:,} ...",
                    flush=True,
                )

            if sample is not None and kept >= sample:
                print(
                    f"      smoke-test sample of {sample:,} notes reached",
                    flush=True,
                )
                break

    finally:
        if writer is not None:
            writer.close()

    if not meta_parts:
        raise RuntimeError(
            "No eligible discharge summaries were found. Check the input "
            "paths and cohort criteria."
        )

    metadata = pd.concat(meta_parts, ignore_index=True)
    del meta_parts
    gc.collect()

    print(f"      scanned {seen:,} total, kept {kept:,}", flush=True)
    print(
        "      median estimated tokens: "
        f"{metadata['n_tokens'].median():,.0f}",
        flush=True,
    )
    print(
        "      section selection disabled; complete notes retained",
        flush=True,
    )

    return metadata


## 7. Define patient-level splitting

Patients, rather than admissions, are assigned to train, validation and test
sets. This prevents different admissions belonging to the same patient from
appearing in multiple splits.


In [7]:
# ============================================================
# CELL 7 - DEFINE PATIENT-LEVEL SPLITTING
# ============================================================


def patient_level_split(df, seed=42, validation=0.15, test=0.15):
    '''Create fixed train/validation/test assignments by subject_id.

    A patient with multiple admissions cannot occur in more than one split.
    Stratification uses the patient's depression/anxiety label combination.
    '''
    rng = np.random.default_rng(seed)

    patient_labels = (
        df.groupby("subject_id")[["depression", "anxiety"]]
        .any()
    )
    patient_labels["stratum"] = (
        patient_labels["depression"].astype(int) * 2
        + patient_labels["anxiety"].astype(int)
    )

    assignments = {}

    for _, stratum in patient_labels.groupby("stratum"):
        subject_ids = stratum.index.to_numpy().copy()
        rng.shuffle(subject_ids)

        number_test = int(len(subject_ids) * test)
        number_validation = int(len(subject_ids) * validation)

        assignments.update(
            {subject_id: "test" for subject_id in subject_ids[:number_test]}
        )
        assignments.update(
            {
                subject_id: "validation"
                for subject_id in subject_ids[
                    number_test:number_test + number_validation
                ]
            }
        )
        assignments.update(
            {
                subject_id: "train"
                for subject_id in subject_ids[
                    number_test + number_validation:
                ]
            }
        )

    return pd.DataFrame(
        {
            "subject_id": list(assignments.keys()),
            "split": list(assignments.values()),
        }
    ).sort_values("subject_id", ignore_index=True)


## 8. Define final assembly and reporting functions


In [8]:
# --------------------------------------------------------------------------
# 5. FINAL ASSEMBLY  -- stream text back in, attach labels, write one file
# --------------------------------------------------------------------------

def assemble(tmp_path, meta, out_path, batch_size=5_000):
    print("[5/6] writing cohort.parquet (text + labels)", flush=True)
    key = meta.set_index("note_id")
    pf = pq.ParquetFile(tmp_path)
    writer = None
    written = 0

    try:
        for batch in pf.iter_batches(batch_size=batch_size,
                                     columns=["note_id", "text"]):
            part = batch.to_pandas()
            # inner join drops duplicate notes not kept for an admission
            part = part.join(key, on="note_id", how="inner")
            if not len(part):
                continue
            part = part[FINAL_COLS]
            table = pa.Table.from_pandas(part, preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(out_path, table.schema,
                                          compression="snappy")
            writer.write_table(table)
            written += len(part)
            del part, table, batch
            gc.collect()
    finally:
        if writer is not None:
            writer.close()
        pf.close()          # Windows will not unlink a file with an open handle

    print(f"      wrote {written:,} rows", flush=True)
    return written


In [9]:
# ============================================================
# CELL 9 - DEFINE AGGREGATE COHORT REPORT
# ============================================================


def build_report(df, splits, matched_codes, n_bipolar_dropped):
    data = df.merge(splits, on="subject_id", how="left")

    report = {
        "n_admissions": int(len(data)),
        "n_patients": int(data["subject_id"].nunique()),
        "n_bipolar_admissions_dropped": int(n_bipolar_dropped),
        "text_representation": "complete_cleaned_discharge_note",
        "section_profile": "none",
        "age": {
            "median": float(data["age_at_admission"].median()),
            "iqr": [
                float(data["age_at_admission"].quantile(0.25)),
                float(data["age_at_admission"].quantile(0.75)),
            ],
            "pct_age_capped_91": round(
                100 * data["age_capped"].mean(),
                2,
            ),
        },
        "sex_pct": (
            data["gender"]
            .value_counts(normalize=True)
            .mul(100)
            .round(2)
            .to_dict()
        ),
        "note_length_tokens_estimated": {
            str(quantile): int(data["n_tokens"].quantile(quantile / 100))
            for quantile in (5, 25, 50, 75, 90, 95, 99)
        },
        "context_limit_reference": {
            "pct_over_512": round(100 * (data["n_tokens"] > 512).mean(), 2),
            "pct_over_1024": round(100 * (data["n_tokens"] > 1024).mean(), 2),
            "pct_over_4096": round(100 * (data["n_tokens"] > 4096).mean(), 2),
        },
        "split_sizes_admissions": data["split"].value_counts().to_dict(),
        "split_sizes_patients": splits["split"].value_counts().to_dict(),
        "matched_icd_codes_top40": matched_codes,
    }

    leakage_columns = {
        "depression": "n_leak_depression",
        "anxiety": "n_leak_anxiety",
    }

    for label in ("depression", "anxiety"):
        positive = data[label].astype(bool)
        own_mention = data[leakage_columns[label]] > 0

        report[label] = {
            "n_positive": int(positive.sum()),
            "prevalence_pct": round(100 * positive.mean(), 2),
            "prevalence_by_split_pct": (
                data.groupby("split")[label]
                .mean()
                .mul(100)
                .round(2)
                .to_dict()
            ),
            "pct_positives_with_explicit_mention": round(
                100 * own_mention[positive].mean(),
                2,
            ),
            "pct_negatives_with_explicit_mention": round(
                100 * own_mention[~positive].mean(),
                2,
            ),
        }

    neither = (
        ~data["depression"].astype(bool)
        & ~data["anxiety"].astype(bool)
    )

    report["undiagnosed"] = {
        "n_negative_for_both": int(neither.sum()),
        "pct_on_antidepressant": round(
            100 * data.loc[neither, "on_antidepressant"].mean(),
            2,
        ),
        "pct_with_any_explicit_mention": round(
            100
            * (
                (
                    data.loc[neither, "n_leak_depression"]
                    + data.loc[neither, "n_leak_anxiety"]
                )
                > 0
            ).mean(),
            2,
        ),
    }

    report["comorbid_both_pct"] = round(
        100
        * (
            data["depression"].astype(bool)
            & data["anxiety"].astype(bool)
        ).mean(),
        2,
    )

    return report


## 9. Validate input files


In [10]:
# ============================================================
# VALIDATE REQUIRED INPUT FILES
# ============================================================

required_files = {
    "patients": locate(HOSP_DIR, "patients"),
    "admissions": locate(HOSP_DIR, "admissions"),
    "diagnoses_icd": locate(HOSP_DIR, "diagnoses_icd"),
    "discharge": locate(NOTE_DIR, "discharge"),
}


input_file_summary = pd.DataFrame(
    [
        {
            "table": table_name,
            "path": file_path,
            "exists": file_path.exists(),
            "size_gb": round(file_path.stat().st_size / (1024 ** 3), 3),
        }
        for table_name, file_path in required_files.items()
    ]
)


display(input_file_summary)


if not input_file_summary["exists"].all():
    raise FileNotFoundError("One or more required MIMIC-IV files are missing.")


print("All required input files were found.")


,table,path,exists,size_gb
0,patients,/Users/purnaperamune/Documents/GitHub/P000464S...,True,0.011
1,admissions,/Users/purnaperamune/Documents/GitHub/P000464S...,True,0.019
2,diagnoses_icd,/Users/purnaperamune/Documents/GitHub/P000464S...,True,0.031
3,discharge,/Users/purnaperamune/Documents/GitHub/P000464S...,True,1.061


All required input files were found.


## 10. Build the older-adult admission cohort


In [11]:
admissions_older = load_admissions(
    HOSP_DIR,
    MIN_AGE,
)


display(admissions_older.head())


print(
    "Older-adult admissions:",
    f"{len(admissions_older):,}",
)


[1/6] patients + admissions
      546,028 admissions -> 287,702 at age >= 60


,subject_id,hadm_id,gender,age_at_admission,age_capped
5,10000084,23052089,M,72,False
6,10000084,29888819,M,72,False
10,10000161,22148160,M,60,False
14,10000635,20642640,F,81,False
15,10000635,26134563,F,74,False


Older-adult admissions: 287,702


## 11. Derive depression and anxiety labels


In [12]:
labels, matched_codes = load_labels(
    HOSP_DIR,
    set(admissions_older["hadm_id"]),
    EXCLUDE_BIPOLAR,
)


display(labels.head())


print(
    "Admissions with at least one diagnoses_icd record:",
    f"{len(labels):,}",
)


[2/6] diagnoses -> binary labels


,hadm_id,depression,anxiety
0,20000019,False,False
1,20000024,False,False
2,20000034,False,False
3,20000041,False,False
4,20000057,False,False


Admissions with at least one diagnoses_icd record: 287,496


## 12. Stream and preserve complete discharge summaries

This is the longest-running step. Every saved `text` value is the complete
lightly cleaned discharge note. Section selection is intentionally deferred to
Notebook 03.


In [13]:
note_metadata = stream_notes(
    NOTE_DIR,
    set(admissions_older["hadm_id"]),
    TEMP_NOTES_PATH,
    chunksize=CHUNK_SIZE,
    tokenizer_name=TOKENIZER_NAME,
    sample=SAMPLE_SIZE,
)


display(note_metadata.head())


[3/6] discharge summaries (streaming)
      text representation: complete cleaned note
      scanned 50,000, kept 28,694 ...
      scanned 100,000, kept 57,523 ...
      scanned 150,000, kept 86,217 ...
      scanned 200,000, kept 115,121 ...
      scanned 250,000, kept 143,981 ...
      scanned 300,000, kept 172,751 ...
      scanned 331,793 total, kept 191,084
      median estimated tokens: 2,212
      section selection disabled; complete notes retained


,note_id,subject_id,hadm_id,note_type,charttime,n_words,char_len,on_antidepressant,n_leak_depression,n_leak_anxiety,n_tokens
0,10000084-DS-17,10000084,23052089,DS,2160-11-25 00:00:00,1396,9395,False,0,0,1954
1,10000764-DS-11,10000764,27897940,DS,2132-10-19 00:00:00,1462,10010,False,0,0,2047
2,10000980-DS-20,10000980,29654838,DS,2188-01-05 00:00:00,1984,13289,False,0,0,2778
3,10000980-DS-21,10000980,26913865,DS,2189-07-03 00:00:00,1755,11786,False,0,0,2457
4,10000980-DS-22,10000980,24947999,DS,2190-11-08 00:00:00,2180,14218,False,1,0,3052


## 13. Keep one complete note per admission and join the cohort

When multiple discharge summaries exist for an admission, the longest
complete cleaned note is retained. `note_id` provides a deterministic
secondary ordering when two notes have the same character length. Missing
depression and anxiety labels are assigned zero, meaning no matching code was
found in `diagnoses_icd`.


In [14]:
# ============================================================
# CELL 13 - KEEP ONE NOTE PER ADMISSION AND JOIN COHORT
# ============================================================

note_metadata = (
    note_metadata
    .sort_values(
        ["char_len", "note_id"],
        ascending=[False, True],
    )
    .drop_duplicates("hadm_id", keep="first")
)


cohort_metadata = (
    admissions_older
    .merge(labels, on="hadm_id", how="left")
    .merge(
        note_metadata.drop(columns=["subject_id"]),
        on="hadm_id",
        how="inner",
    )
)


number_bipolar_dropped = 0


if EXCLUDE_BIPOLAR:
    cohort_metadata["bipolar"] = (
        cohort_metadata["bipolar"]
        .fillna(False)
        .astype(bool)
    )
    number_bipolar_dropped = int(cohort_metadata["bipolar"].sum())
    cohort_metadata = (
        cohort_metadata.loc[~cohort_metadata["bipolar"]]
        .drop(columns=["bipolar"])
        .copy()
    )


for label_column in ("depression", "anxiety"):
    cohort_metadata[label_column] = (
        cohort_metadata[label_column]
        .fillna(False)
        .astype(int)
    )


display(
    cohort_metadata[
        [
            "hadm_id",
            "subject_id",
            "age_at_admission",
            "gender",
            "depression",
            "anxiety",
            "n_tokens",
        ]
    ].head()
)


print("Cohort admissions:", f"{len(cohort_metadata):,}")
print("Cohort patients:", f"{cohort_metadata['subject_id'].nunique():,}")


,hadm_id,subject_id,age_at_admission,gender,depression,anxiety,n_tokens
0,23052089,10000084,72,M,0,0,1954
1,27897940,10000764,86,M,0,0,2047
2,20897796,10000980,80,F,0,0,3146
3,24947999,10000980,77,F,0,0,3052
4,25242409,10000980,78,F,0,0,2985


Cohort admissions: 191,084
Cohort patients: 80,087


## 14. Create and save patient-level splits


In [15]:
splits = patient_level_split(
    cohort_metadata,
    seed=SEED,
)


splits.to_parquet(
    OUTPUT_DIR / "splits.parquet",
    index=False,
)


split_summary = (
    splits["split"]
    .value_counts()
    .rename_axis("split")
    .reset_index(name="patients")
)


display(split_summary)


,split,patients
0,train,56065
1,test,12011
2,validation,12011


## 15. Write the final cohort and audit reports


In [16]:
# ============================================================
# CELL 15 - WRITE COHORT AND AUDIT REPORTS
# ============================================================

number_written = assemble(
    TEMP_NOTES_PATH,
    cohort_metadata,
    OUTPUT_DIR / "cohort.parquet",
)


cohort_report = build_report(
    cohort_metadata,
    splits,
    matched_codes,
    number_bipolar_dropped,
)


(OUTPUT_DIR / "cohort_stats.json").write_text(
    json.dumps(
        cohort_report,
        indent=2,
        default=str,
    )
)


histogram_counts, histogram_edges = np.histogram(
    cohort_metadata["n_tokens"],
    bins=50,
)


pd.DataFrame(
    {
        "bin_left": histogram_edges[:-1],
        "bin_right": histogram_edges[1:],
        "count": histogram_counts,
    }
).to_csv(
    OUTPUT_DIR / "note_length_hist.csv",
    index=False,
)


if not KEEP_TEMPORARY_FILE:
    try:
        TEMP_NOTES_PATH.unlink(missing_ok=True)
    except OSError as error:
        print(
            "Temporary file could not be removed:",
            error,
        )


print("Cohort rows written:", f"{number_written:,}")
print("Output directory:", OUTPUT_DIR)


[5/6] writing cohort.parquet (text + labels)
      wrote 191,084 rows
Cohort rows written: 191,084
Output directory: /Users/purnaperamune/Documents/GitHub/P000464SE-Modern-Transformer-Models-for-Depression-Anxiety-Detection/dataset/cohort_output


## 16. Inspect and validate the outputs

Only Parquet metadata and the first five cohort records are read. The complete
cohort is not reloaded solely for previewing. The `text` column remains
readable complete text; `n_tokens` is only a length estimate.


In [17]:
# ============================================================
# CELL 16 - INSPECT AND VALIDATE OUTPUTS
# ============================================================

output_files = [
    OUTPUT_DIR / "cohort.parquet",
    OUTPUT_DIR / "splits.parquet",
    OUTPUT_DIR / "cohort_stats.json",
    OUTPUT_DIR / "note_length_hist.csv",
]


output_summary = pd.DataFrame(
    {
        "file": [path.name for path in output_files],
        "exists": [path.exists() for path in output_files],
        "size_mb": [
            round(path.stat().st_size / (1024 ** 2), 3)
            if path.exists()
            else np.nan
            for path in output_files
        ],
    }
)


display(output_summary)


cohort_file = pq.ParquetFile(
    OUTPUT_DIR / "cohort.parquet"
)


cohort_preview = next(
    cohort_file.iter_batches(
        batch_size=5,
    )
).to_pandas()


display(cohort_preview)


if cohort_preview["text"].isna().any():
    raise ValueError("The cohort preview contains missing text.")


if not set(cohort_preview["depression"].unique()).issubset({0, 1}):
    raise ValueError("Unexpected depression label detected.")


if not set(cohort_preview["anxiety"].unique()).issubset({0, 1}):
    raise ValueError("Unexpected anxiety label detected.")


cohort_file.close()


print("Notebook 02 completed successfully.")
print("Complete discharge text was preserved for Notebook 03.")


,file,exists,size_mb
0,cohort.parquet,True,1049.205
1,splits.parquet,True,0.522
2,cohort_stats.json,True,0.003
3,note_length_hist.csv,True,0.001


,hadm_id,subject_id,age_at_admission,gender,depression,anxiety,n_tokens,text
0,23052089,10000084,72,M,0,0,1954,Name: [DEID] Unit No: [DEID]\n \nAdmission Dat...
1,27897940,10000764,86,M,0,0,2047,Name: [DEID] Unit No: [DEID]\n \nAdmission Dat...
2,29654838,10000980,75,F,0,0,2778,Name: [DEID] Unit No: [DEID]\n \nAdmission Dat...
3,26913865,10000980,76,F,0,0,2457,Name: [DEID] Unit No: [DEID]\n \nAdmission Dat...
4,24947999,10000980,77,F,0,0,3052,Name: [DEID] Unit No: [DEID]\n \nAdmission Dat...


Notebook 02 completed successfully.
Complete discharge text was preserved for Notebook 03.
